# Clase 204 — Shadow + Canary + A/B test

Simulamos los 3 patrones en proceso (sin Istio), con sticky assignment, métricas, rollback automático y análisis estadístico del A/B.

## Setup — 2 modelos: champion vs challenger

In [ ]:
import numpy as np, hashlib, time, random
from collections import defaultdict
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X, y = load_breast_cancer(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42)

champion = LogisticRegression(max_iter=5000).fit(Xtr, ytr)
challenger = RandomForestClassifier(n_estimators=200, random_state=42).fit(Xtr, ytr)
print(f'champion offline acc: {accuracy_score(yte, champion.predict(Xte)):.4f}')
print(f'challenger offline acc: {accuracy_score(yte, challenger.predict(Xte)):.4f}')

## 1. Shadow mode (challenger no responde)

In [ ]:
shadow_log = []

def predict_with_shadow(row, true_label):
    t0 = time.perf_counter()
    champ_pred = int(champion.predict(row.reshape(1, -1))[0])
    t_champ = time.perf_counter() - t0
    t1 = time.perf_counter()
    chal_pred = int(challenger.predict(row.reshape(1, -1))[0])
    t_chal = time.perf_counter() - t1
    shadow_log.append({
        'champ_pred': champ_pred, 'chal_pred': chal_pred, 'truth': int(true_label),
        'champ_lat_ms': t_champ * 1000, 'chal_lat_ms': t_chal * 1000,
    })
    return champ_pred   # solo champion responde al usuario

for i in range(len(Xte)):
    predict_with_shadow(Xte[i], yte[i])

import pandas as pd
log_df = pd.DataFrame(shadow_log)
agreement = (log_df.champ_pred == log_df.chal_pred).mean()
champ_acc = (log_df.champ_pred == log_df.truth).mean()
chal_acc = (log_df.chal_pred == log_df.truth).mean()
print(f'agreement champion-challenger: {agreement:.2%}')
print(f'accuracy champion:   {champ_acc:.4f}')
print(f'accuracy challenger: {chal_acc:.4f}  ({"🟢 better" if chal_acc > champ_acc else "🔴 worse"})')
print(f'latency champion:   p99={log_df.champ_lat_ms.quantile(0.99):.2f} ms')
print(f'latency challenger: p99={log_df.chal_lat_ms.quantile(0.99):.2f} ms')

## 2. Canary release con sticky assignment

In [ ]:
def sticky_bucket(user_id: str, percent: int) -> str:
    """Hash determinista → 'challenger' si está en el primer `percent`% del espacio."""
    h = int(hashlib.md5(user_id.encode()).hexdigest(), 16) % 100
    return 'challenger' if h < percent else 'champion'

# Simular 1000 users con canary al 10%
metrics = defaultdict(lambda: {'n': 0, 'correct': 0, 'lat_sum_ms': 0})
for i in range(1000):
    user_id = f'user_{i}'
    bucket = sticky_bucket(user_id, percent=10)
    model = challenger if bucket == 'challenger' else champion
    row = Xte[i % len(Xte)]
    t0 = time.perf_counter()
    pred = int(model.predict(row.reshape(1, -1))[0])
    lat = (time.perf_counter() - t0) * 1000
    truth = int(yte[i % len(yte)])
    metrics[bucket]['n'] += 1
    metrics[bucket]['correct'] += int(pred == truth)
    metrics[bucket]['lat_sum_ms'] += lat

for b in ['champion', 'challenger']:
    m = metrics[b]
    print(f'{b}: n={m["n"]}, acc={m["correct"] / m["n"]:.4f}, lat_mean={m["lat_sum_ms"] / m["n"]:.2f} ms')

## 3. Auto-rollback simulado

In [ ]:
def health_check_and_maybe_rollback(metrics_champion, metrics_challenger, lat_p99_tolerance=1.2, err_max=0.05):
    """Devuelve nuevo canary % (0 = rollback)."""
    c_lat = metrics_champion['lat_p99_ms']
    h_lat = metrics_challenger['lat_p99_ms']
    h_err = metrics_challenger['error_rate']
    if h_lat > c_lat * lat_p99_tolerance:
        return 0, f'ROLLBACK: lat_p99 challenger {h_lat:.1f} > {lat_p99_tolerance}× champion {c_lat:.1f}'
    if h_err > err_max:
        return 0, f'ROLLBACK: error rate challenger {h_err:.2%} > {err_max:.2%}'
    return None, 'OK'

# Caso 1: challenger sano
print(health_check_and_maybe_rollback({'lat_p99_ms': 8, 'error_rate': 0.01}, {'lat_p99_ms': 9, 'error_rate': 0.012}))
# Caso 2: challenger lento
print(health_check_and_maybe_rollback({'lat_p99_ms': 8, 'error_rate': 0.01}, {'lat_p99_ms': 25, 'error_rate': 0.01}))
# Caso 3: challenger con errores
print(health_check_and_maybe_rollback({'lat_p99_ms': 8, 'error_rate': 0.01}, {'lat_p99_ms': 9, 'error_rate': 0.08}))

## 4. A/B test riguroso — sample size y p-value

In [ ]:
from scipy import stats

def sample_size_for_proportions(p1, p2, alpha=0.05, power=0.8):
    """Tamaño por arm para detectar diff p2-p1 con dado poder."""
    z_alpha = stats.norm.ppf(1 - alpha / 2)
    z_beta = stats.norm.ppf(power)
    p_bar = (p1 + p2) / 2
    n = ((z_alpha * (2 * p_bar * (1 - p_bar)) ** 0.5 + z_beta * (p1 * (1 - p1) + p2 * (1 - p2)) ** 0.5) ** 2) / (p2 - p1) ** 2
    return int(np.ceil(n))

n_per_arm = sample_size_for_proportions(p1=0.92, p2=0.94)
print(f'sample size para detectar 92% → 94% con α=0.05 power=0.8: {n_per_arm} por arm')

# Simulamos los resultados con N exacto
rng = np.random.default_rng(0)
champ_outcomes = rng.binomial(1, 0.92, n_per_arm)
chal_outcomes = rng.binomial(1, 0.94, n_per_arm)
from statsmodels.stats.proportion import proportions_ztest
z, p = proportions_ztest([chal_outcomes.sum(), champ_outcomes.sum()], [n_per_arm, n_per_arm])
print(f'observado: champ {champ_outcomes.mean():.4f}, chal {chal_outcomes.mean():.4f}')
print(f'z={z:.3f}, p={p:.4f}  →  {"SIGNIFICATIVO" if p < 0.05 else "NO significativo"}')

## Ejercicio guiado

1. Convertí el shadow log a un análisis de **disagreement triage**: para las filas donde champion y challenger difieren, ¿cuál acierta más?
2. Implementá canary progresivo: empezá 1% → 5% → 25% → 100% con health checks entre cada escalón.
3. Agregá un guardrail: si `business_kpi_proxy` (ej. `proba > 0.7` rate) cae > 10%, no escales el canary.
4. Calculá sample size para tu caso real: detectar 1 punto de diff con `power=0.8`. ¿Cuántos días de tráfico necesitás?
5. Bonus: implementá esto en Istio con `VirtualService` y verificá distribución real de tráfico.

## Conclusiones

- Shadow = sin riesgo, costo 2×. Canary = riesgo limitado, costo 1×.
- Sticky assignment es no-negociable; sin él, A/B test es ruido.
- Rollback automático debe ser parte del deploy, no "plan B manual".
- Sample size pre-calculado vs ad-hoc es la diferencia entre decisión basada en evidencia y theater.

## ✅ Soluciones de los ejercicios

Soluciones de los 5 ejercicios del README. Casi todo es **ejecutable con la base científica**: shadow in-process, routing canary con hash sticky, rollback por métrica y el tamaño de muestra de un A/B test. Istio (ex3) es infra externa: mostramos el `VirtualService` real y simulamos su routing por pesos con numpy.

In [ ]:
import numpy as np, hashlib
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=4000, n_features=10, random_state=1)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.5, random_state=1)
champion = LogisticRegression(max_iter=500).fit(Xtr, ytr)
challenger = RandomForestClassifier(n_estimators=60, random_state=0).fit(Xtr, ytr)
print('champion (LogReg) y challenger (RF) entrenados.')

### Ejercicio 1 — Shadow deployment en proceso

Se predice con ambos modelos, se **devuelve solo el champion** y se loguean los dos. El challenger no afecta al usuario; sirve para comparar su distribución de predicciones contra el champion sobre tráfico real.

In [ ]:
logs = []
def serve_with_shadow(x):
    x = x.reshape(1, -1)
    champ = int(champion.predict(x)[0])
    chall = int(challenger.predict(x)[0])         # shadow: se calcula pero NO se devuelve
    logs.append({'champion': champ, 'challenger': chall})
    return champ                                   # el usuario ve solo el champion

for row in Xte[:1000]:
    serve_with_shadow(row)

agree = np.mean([l['champion'] == l['challenger'] for l in logs])
print(f'requests en shadow: {len(logs)} | acuerdo champion-challenger: {agree:.1%}')
assert len(logs) == 1000 and 0 <= agree <= 1
print('OK — shadow mide el challenger en producción sin exponer al usuario al riesgo.')

### Ejercicio 2 — Canary con feature flag + sticky por user_id

`CANARY_PERCENT` decide qué fracción va al challenger. **Sticky**: hasheamos el `user_id`, así el mismo usuario ve siempre el mismo modelo dentro del test (consistencia de experiencia).

In [ ]:
def route(user_id, canary_percent):
    bucket = int(hashlib.sha256(str(user_id).encode()).hexdigest(), 16) % 100
    return 'challenger' if bucket < canary_percent else 'champion'

# stickiness: el mismo user_id -> siempre el mismo modelo
assert route(42, 5) == route(42, 5) == route(42, 5), 'debe ser sticky'
# proporción ~ canary_percent sobre muchos usuarios
routes = [route(uid, 5) for uid in range(10000)]
frac = routes.count('challenger') / len(routes)
print(f'fracción al challenger con CANARY_PERCENT=5: {frac:.1%} (esperado ~5%)')
assert 0.03 < frac < 0.07, 'la proporción debe aproximar el porcentaje configurado'
print('OK — canary controlado por env var, sticky por hash de user_id.')

### Ejercicio 3 — Canary con Istio (VirtualService)

Istio reparte tráfico por pesos a nivel de malla. Mostramos el `VirtualService` 95/5 y simulamos su routing con numpy para verificar la proporción.

In [ ]:
import yaml
virtual_service = '''
apiVersion: networking.istio.io/v1beta1
kind: VirtualService
metadata: {name: iris-api}
spec:
  hosts: [iris-api]
  http:
    - route:
        - destination: {host: iris-api, subset: champion}
          weight: 95
        - destination: {host: iris-api, subset: challenger}
          weight: 5
'''
vs = yaml.safe_load(virtual_service)
weights = {r['destination']['subset']: r['weight'] for r in vs['spec']['http'][0]['route']}
assert weights['champion'] + weights['challenger'] == 100
rng = np.random.default_rng(0)
picks = rng.choice(['champion', 'challenger'], size=100000, p=[weights['champion']/100, weights['challenger']/100])
frac = np.mean(picks == 'challenger')
print(f'pesos: {weights} | fracción challenger simulada: {frac:.2%}')
assert abs(frac - 0.05) < 0.01
print('OK — kubectl apply -f vs.yaml enruta ~5% al challenger sin tocar la app.')

### Ejercicio 4 — Rollback automático por métrica

Un sidecar consulta Prometheus cada 60 s; si la `latency_p99` del challenger supera el umbral, mueve el peso a `100/0` (rollback). Simulamos la lógica de decisión.

In [ ]:
def rollback_controller(p99_ms, threshold_ms=200, weights=(95, 5)):
    if p99_ms > threshold_ms:
        return (100, 0)                # rollback: todo al champion
    return weights

assert rollback_controller(150) == (95, 5), 'dentro de SLO: mantiene el canary'
assert rollback_controller(250) == (100, 0), 'fuera de SLO: rollback automático'
print('p99=150ms ->', rollback_controller(150), '| p99=250ms ->', rollback_controller(250))
print('OK — el guardrail corta el experimento antes de que degrade a los usuarios.')

### Ejercicio 5 — A/B test riguroso: tamaño de muestra + decisión

Calculamos el N necesario para detectar `δ=0.02` en accuracy (`α=0.05`, `power=0.8`) con la fórmula de dos proporciones, corremos el test simulado y reportamos p-value + IC de la diferencia.

In [ ]:
from scipy import stats as st

def sample_size_two_proportions(p1, delta, alpha=0.05, power=0.8):
    p2 = p1 + delta
    z_a = st.norm.ppf(1 - alpha / 2)
    z_b = st.norm.ppf(power)
    pbar = (p1 + p2) / 2
    n = (z_a * np.sqrt(2 * pbar * (1 - pbar)) + z_b * np.sqrt(p1*(1-p1) + p2*(1-p2))) ** 2 / delta**2
    return int(np.ceil(n))

n = sample_size_two_proportions(p1=0.85, delta=0.02)
print(f'N por grupo para detectar δ=0.02: {n:,}')

rng = np.random.default_rng(3)
a = rng.binomial(1, 0.85, n)          # champion
b = rng.binomial(1, 0.87, n)          # challenger (mejor por δ)
# test de dos proporciones (z)
pa, pb = a.mean(), b.mean()
diff = pb - pa
se = np.sqrt(pa*(1-pa)/n + pb*(1-pb)/n)
z = diff / se
pval = 2 * (1 - st.norm.cdf(abs(z)))
ci = (diff - 1.96*se, diff + 1.96*se)
print(f'accuracy champion={pa:.3f} challenger={pb:.3f} | Δ={diff:+.3f}')
print(f'p-value={pval:.4f} | IC95%=({ci[0]:+.3f}, {ci[1]:+.3f})')
assert n > 1000 and se > 0
print('OK — decisión basada en N adecuado, p-value e IC (no en “se ve mejor”).')